## Prerequisites

Before running this notebook:

1. Authenticate with Databricks:
```bash
databricks auth login
```

2. Prepare the data:
```bash
uv run --group databricks python -m integration.databricks.prepare_jaffle_shop_data_for_databricks
```

This will load raw data from GCS and create the weekly sales forecasting population table.

## Setup and Data Loading

In [ ]:
from pathlib import Path

import getml
from databricks.connect import DatabricksSession

PROJECT_NAME = "databricks_feature_store"

getml.set_project(PROJECT_NAME)

  Loading pipelines...                     ━━━━━━━━━━━━━━━ 100% • 00:00


Connected to project 'databricks_feature_store'.

In [33]:
spark = DatabricksSession.builder.serverless().getOrCreate()

In [ ]:
# Load population table
weekly_sales_by_store_spark = spark.table(
    "workspace.prepared.weekly_sales_by_store_with_target"
)

weekly_sales_by_store = getml.DataFrame.from_arrow(
    weekly_sales_by_store_spark.toArrow(), name="weekly_sales_by_store"
)

# Load peripheral table
orders_spark = spark.table("workspace.raw.raw_orders")
orders = getml.DataFrame.from_arrow(orders_spark.toArrow(), name="orders")

/Users/awais/code17/projects/getML/github/getml-demo/.venv/lib/python3.12/site-packages/getml/data/_io/arrow.py:371: UserWarning:     
Column 'next_week_sales' has been converted from decimal to float. This may
    result in a loss of precision!
    
  warnings.warn(


## getML Annotations

In [5]:
weekly_sales_by_store.set_role(
    cols=["store_id", "snapshot_id"], role=getml.data.roles.join_key
)
weekly_sales_by_store.set_role(cols="reference_date", role=getml.data.roles.time_stamp)
weekly_sales_by_store.set_role(cols="next_week_sales", role=getml.data.roles.target)
weekly_sales_by_store.set_role(
    cols=[
        "store_name",
        "year",
        "month",
        "week_number",
        "is_full_week_after_opening",
        "has_order_activity",
        "has_min_history",
    ],
    role=getml.data.roles.categorical,
)
weekly_sales_by_store.set_role(
    cols=["days_since_open", "next_week_orders"], role=getml.data.roles.numerical
)
weekly_sales_by_store


/Users/awais/code17/projects/getML/github/getml-demo/.venv/lib/python3.12/site-packages/getml/data/_io/arrow.py:393: UserWarning:     
Column 'reference_date' has UTC timezone. Dropping the timezone.

    Currently, getML doesn't support the handling of explicit timezones for
timestamps. If you need timezone information to be available for your models, you
can encode the timezone explicitly in a separate column.
    
  warnings.warn(


name,reference_date,store_id,snapshot_id,next_week_sales,store_name,year,month,week_number,is_full_week_after_opening,has_order_activity,has_min_history,days_since_open,next_week_orders
role,time_stamp,join_key,join_key,target,categorical,categorical,categorical,categorical,categorical,categorical,categorical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,
0,2022-09-19,eafbd328-0434-4f46-9c2d-cc97a46f...,778,26020.74,Brooklyn,2022,9,38,true,true,true,1287,2389
1,2023-10-16,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1115,19807.07,Philadelphia,2023,10,42,true,true,true,1871,1627
2,2024-07-15,eafbd328-0434-4f46-9c2d-cc97a46f...,1348,21198.29,Brooklyn,2024,7,29,true,true,true,1952,2066
3,2023-08-14,eafbd328-0434-4f46-9c2d-cc97a46f...,1060,20418.27,Brooklyn,2023,8,33,true,true,true,1616,2054
4,2020-01-27,eafbd328-0434-4f46-9c2d-cc97a46f...,119,25682.76,Brooklyn,2020,1,5,true,true,true,321,2382
,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,2022-08-29,abfcc332-1eaf-42f6-b8e4-569bf6a3...,758,18024.09,Chicago,2022,8,35,true,true,true,853,1685
1375,2024-04-08,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1265,18632.58,Philadelphia,2024,4,15,true,true,true,2046,1589


In [6]:
orders.set_role(cols=["store_id", "id", "customer"], role=getml.data.roles.join_key)
orders.set_role(
    cols="ordered_at",
    role=getml.data.roles.time_stamp,
    time_formats=["%Y-%m-%dT%H:%M:%S"],
)
orders.set_role(
    cols=["subtotal", "order_total", "tax_paid"], role=getml.data.roles.numerical
)
orders

name,ordered_at,store_id,id,customer,subtotal,order_total,tax_paid
role,time_stamp,join_key,join_key,join_key,numerical,numerical,numerical
unit,"time stamp, comparison only",,,,,,
0,2018-09-01 09:07:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,2dfe5624-4340-4ae0-af9f-01edf04c...,304f8066-3937-42ab-9edc-5e2c937e...,4000,4240,240
1,2018-09-01 08:32:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,80c27a4c-7d23-49fe-864e-1859a166...,ab45b9b0-e8cb-4eed-b1c7-5b3a70cd...,500,530,30
2,2018-09-01 13:00:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,bd7d415c-a44e-47ba-bf35-727ecf8b...,530d100f-15de-49ea-93bb-28b570a7...,3200,3392,192
3,2018-09-01 11:14:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,ed02759f-4d40-44c5-8583-2eddaf1e...,5603ee30-171b-4f23-a0b3-3b4cd282...,5300,5618,318
4,2018-09-01 12:49:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,ff48d20e-7294-4ca1-9baa-85821bda...,5512b3a7-632d-4daa-94a3-d24533fc...,600,636,36
,...,...,...,...,...,...,...
2309598,2024-08-29 13:01:00,61743c9b-2394-4d36-8062-8a6820fa...,479b3bbd-170c-4fe6-9b83-19e9dd15...,6ab94c27-8718-4907-854b-c8a4892e...,400,432,32
2309599,2024-08-29 10:27:00,61743c9b-2394-4d36-8062-8a6820fa...,8e1feeb5-b8a6-47fc-8432-f8b7aea3...,a866ff38-808f-458d-9c15-bb87b383...,600,648,48


## getML Data Model

In [8]:
validation_begin = getml.data.time.datetime(2023, 1, 1)
test_begin = getml.data.time.datetime(2024, 1, 1)

split = getml.data.split.time(
    population=weekly_sales_by_store,
    time_stamp="reference_date",
    validation=validation_begin,
    test=test_begin,
)

# Filter dataframes using the split column
weekly_sales_by_store_train = weekly_sales_by_store[split == "train"]
weekly_sales_by_store_validation = weekly_sales_by_store[split == "validation"]
weekly_sales_by_store_test = weekly_sales_by_store[split == "test"]

print(
    f"Training set size: {len(weekly_sales_by_store_train)}"
    f"\nValidation set size: {len(weekly_sales_by_store_validation)}"
    f"\nTest set size: {len(weekly_sales_by_store_test)}"
)

Training set size: 863
Validation set size: 312
Test set size: 204


In [38]:
data_model = getml.data.DataModel(
    population=weekly_sales_by_store_train.to_placeholder("weekly_sales_by_store")
)

# Add all peripheral tables
data_model.add(
    getml.data.to_placeholder(
        orders=orders,
    )
)

# Define relationships using joins
data_model.weekly_sales_by_store.join(
    right=data_model.orders,
    on="store_id",
    time_stamps=("reference_date", "ordered_at"),
    relationship=getml.data.relationship.one_to_many,
    memory=getml.data.time.days(30),
)

In [ ]:
container = getml.data.Container(
    train=weekly_sales_by_store_train,
    validation=weekly_sales_by_store_validation,
    test=weekly_sales_by_store_test,
)

# Add peripheral tables with aliases matching the data model placeholders
container.add(
    orders=orders,
)

getml.project.data_frames.save()
container.save()

In [40]:
container._id

'uzNVUz'

## Training

In [9]:
fast_prop = getml.feature_learning.FastProp()

predictor = getml.predictors.XGBoostRegressor(
    n_jobs=0,
)

pipe = getml.Pipeline(
    data_model=data_model,
    feature_learners=[
        fast_prop,
    ],
    predictors=[predictor],
)

pipe.fit(container.train)
pipe.score(container.test)

Checking data model...

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Checking...                              ━━━━━━━━━━━━━━━ 100% • 00:00


The pipeline check generated 0 issues labeled INFO and 2 issues labeled WARNING.

To see the issues in full, run .check() on the pipeline.

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:01
  FastProp: Trying 50 features...          ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:02
  XGBoost: Training as predictor...        ━━━━━━━━━━━━━━━ 100% • 00:02


Trained pipeline.

Time taken: 0:00:06.494403.

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:01


,date time,set used,target,mae,rmse,rsquared
0,2026-01-06 15:39:45,train,next_week_sales,249.6961,319.0279,0.9973
1,2026-01-06 15:39:49,test,next_week_sales,472.4813,613.3892,0.9862


In [ ]:
# container = getml.data.load_container("uzNVUz")
# pipe = getml.pipeline.load("2aj2Dm")

## Feature Export

In [13]:
# Transform features using getML pipeline
features_df = pipe.transform(
    population_table=weekly_sales_by_store,
    peripheral_tables=[orders],
    df_name="features",
)


features_spark = spark.createDataFrame(features_df.to_arrow())

features_df

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00


name,reference_date,store_id,snapshot_id,next_week_sales,feature_1_1,feature_1_2,feature_1_3,feature_1_4,feature_1_5,feature_1_6,feature_1_7,feature_1_8,feature_1_9,feature_1_10,feature_1_11,feature_1_12,feature_1_13,feature_1_14,feature_1_15,feature_1_16,feature_1_17,feature_1_18,feature_1_19,feature_1_20,feature_1_21,feature_1_22,feature_1_23,feature_1_24,feature_1_25,feature_1_26,feature_1_27,feature_1_28,feature_1_29,feature_1_30,feature_1_31,feature_1_32,feature_1_33,feature_1_34,feature_1_35,feature_1_36,feature_1_37,feature_1_38,feature_1_39,feature_1_40,feature_1_41,feature_1_42,feature_1_43,feature_1_44,feature_1_45,feature_1_46,feature_1_47,feature_1_48,feature_1_49,feature_1_50,days_since_open,next_week_orders
role,time_stamp,join_key,join_key,target,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,2022-09-19,eafbd328-0434-4f46-9c2d-cc97a46f...,778,26020.74,1032.1334,85,8193,1800,3600,9800,600,0,600,1186.2928,8544000,1065.3569,1073.4165,85,8193,1872,3744,10192,624,0,624,1233.7408,8885742,1107.9689,41.2832,85,8193,72,144,392,24,0,24,47.448,341742,42.612,1295769.6183,5554,2724,2563200,32460,2563200,1255740,32460,234000,709850.1638,10726380900,0,305.7557,8278,1287,2389
1,2023-10-16,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1115,19807.07,1140.2708,88,6264,700,1900,9800,600,0,600,1274.3324,7243000,1163.8545,1208.62,88,6264,742,2013,10388,636,0,636,1350.7264,7677154,1233.6033,68.3492,88,6264,42,113,588,36,0,36,76.3944,434154,69.7488,1258149.0397,4899,1453,2562060,32520,2562060,1181490,32520,2221200,699016.8555,7991762700,0,398.29,6352,1871,1627
2,2024-07-15,eafbd328-0434-4f46-9c2d-cc97a46f...,1348,21198.29,1059.0929,85,8845,1300,600,9500,600,0,600,1216.411,9457700,1043.2765,1101.4541,85,8845,1352,624,9880,624,0,624,1265.0631,9835985,1085.0047,42.3611,85,8845,52,24,380,24,0,24,48.652,378285,41.7281,1347191.953,5990,2940,2563200,32700,2563200,1423410,32700,2394000,728065.9826,12030424140,0,283.4024,8930,1952,2066
3,2023-08-14,eafbd328-0434-4f46-9c2d-cc97a46f...,1060,20418.27,1046.0875,84,8376,1100,8900,9700,600,0,600,1228.2472,8849900,1061.4084,1087.9287,84,8376,1144,9256,10088,624,0,624,1277.3731,9203877,1103.8616,41.8413,84,8376,44,356,388,24,0,24,49.126,353977,42.4532,1294489.4965,5676,2784,2563200,33300,2563200,1264200,33300,1530000,712443.903,10951381140,0,299.0779,8460,1616,2054
4,2020-01-27,eafbd328-0434-4f46-9c2d-cc97a46f...,119,25682.76,1047.4757,82,9386,700,8400,9500,600,0,600,1106.4951,9917500,1064.6682,1089.373,82,9386,728,8736,9880,624,0,624,1150.7521,10314184,1107.2526,41.8973,82,9386,28,336,380,24,0,24,44.257,396684,42.5844,1289085.9696,6631,2837,2560560,32580,2560560,1251690,32580,1530000,706466.9565,12205065960,0,267.0307,9468,321,2382
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,2022-08-29,abfcc332-1eaf-42f6-b8e4-569bf6a3...,758,18024.09,1075.6416,84,6969,3400,1800,9600,600,0,600,1251.6585,7586500,1130.5452,1142.4785,84,6969,3612,1912,10200,637,0,637,1329.8809,8057901,1200.8109,66.8369,84,6969,212,112,600,37,0,37,78.2229,471401,70.2656,1296983.6835,5047,2006,2563200,32520,2563200,1256640,32520,1530000,715254.7768,9147625920,0,358.8599,7053,853,1685
1375,2024-04-08,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1265,18632.58,1163.6744,86,6598,5500,1600,9600,600,0,600,1302.6833,7778000,1165.

In [28]:
from getml_interpretations import (
    ColumnDescriptionsReport,
    generate_column_descriptions_report,
)

column_descriptions_report_path = Path("column_descriptions_report.json")
jaffle_shop_annotations = Path("annotations.yml")
user_prompt = f"""
Please use the following annotations as source of truth for generating the
column descriptions report:

{jaffle_shop_annotations.read_text()}.
"""

if column_descriptions_report_path.exists():
    column_descriptions_report = ColumnDescriptionsReport.from_json(
        column_descriptions_report_path
    )
else:
    column_descriptions_report: ColumnDescriptionsReport = (
        await generate_column_descriptions_report(
            project_name=PROJECT_NAME,
            pipeline=pipe,
            container=container,
            model="gpt-5-mini",
            user_prompt=user_prompt,
        )
    )
    column_descriptions_report.to_json(column_descriptions_report_path)


In [15]:
from pathlib import Path

from getml_interpretations import (
    FeatureDescriptionsReport,
    generate_feature_descriptions_report,
)

feature_descriptions_report_path = Path("feature_descriptions_report.json")

user_prompt = """
Feature descriptions are limited to max. 256 characters and 2-3 sentences.
Please ensure that each feature description does not exceed this limit.
Also no bullet points, new lines or line breaks etc.
Keep the descriptions concise, straight-forward and informative.
"""

if feature_descriptions_report_path.exists():
    feature_descriptions_report = FeatureDescriptionsReport.from_json(
        feature_descriptions_report_path
    )
else:
    feature_descriptions_report: FeatureDescriptionsReport = (
        await generate_feature_descriptions_report(
            project_name=PROJECT_NAME,
            pipeline=pipe,
            container=container,
            user_prompt=user_prompt,
            model="gpt-5-mini",
            batch_size=20,
        )
    )
    feature_descriptions_report.to_json(feature_descriptions_report_path)

In [ ]:
column_mapping = {
    desc.name: desc.title
    for desc in feature_descriptions_report.feature_descriptions.values()
}

for old_name, new_name in column_mapping.items():
    features_spark = features_spark.withColumnRenamed(old_name, new_name)


In [24]:
FEATURES_TABLE_FULL_NAME = "workspace.getml_fs.getml_features_explained"
spark.sql(f"DROP TABLE IF EXISTS {FEATURES_TABLE_FULL_NAME}")

features_spark.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(FEATURES_TABLE_FULL_NAME)

spark.sql(f"""
    ALTER TABLE {FEATURES_TABLE_FULL_NAME}
    ALTER COLUMN snapshot_id SET NOT NULL
""")

FEATURES_TABLE_NAME = FEATURES_TABLE_FULL_NAME.split(".")[-1]

spark.sql(f"""
    ALTER TABLE {FEATURES_TABLE_FULL_NAME}
    ADD CONSTRAINT {FEATURES_TABLE_NAME}_pk PRIMARY KEY(snapshot_id)
""")

DataFrame[]

In [39]:
for tables in column_descriptions_report.column_descriptions.values():
    for column_name, column_description in tables.items():
        if column_name in features_df.colnames:
            # Escape single quotes to prevent SQL syntax errors
            description = column_description.description.replace("'", "\\'")
            spark.sql(
                f"ALTER TABLE {FEATURES_TABLE_FULL_NAME} CHANGE COLUMN {column_name} COMMENT '{description}'"
            )

Updating description for column: reference_date
Updating description for column: store_id
Updating description for column: snapshot_id
Updating description for column: next_week_sales
Updating description for column: days_since_open
Updating description for column: next_week_orders
Updating description for column: store_id


In [34]:
for feature_description in feature_descriptions_report.feature_descriptions.values():
    # Escape single quotes to prevent SQL syntax errors
    description = feature_description.description.replace("'", "\\'")
    spark.sql(
        f"ALTER TABLE {FEATURES_TABLE_FULL_NAME} CHANGE COLUMN {feature_description.title} COMMENT '{description}'"
    )


In [19]:
# Verify the Feature Table was created
print("Table schema:")
spark.sql(f"DESCRIBE TABLE {FEATURES_TABLE_FULL_NAME}").show(100, truncate=False)


Table schema:
+-------------------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------